In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# PPO in detail

PPO is one of the earliest (and thus, the "default") Rl method for post-training LLMs.

It has a lot of pieces which may make it intimidating.

In this module, we try to explain PPO in detail.



# Components

| Persona | Mathematical Signature | Role in the Loop | Status |
| :--- | :--- | :--- | :--- |
| **Actor** | $\pi_\theta(s_t) \rightarrow P(a_t)$ | **The Performer:** Samples the next token. This is the final model. | **Active** (Updates) |
| **Old Actor** | $\pi_{old}(s_t) \rightarrow P(a_t)$ | **The Ghost:** A frozen snapshot of the Actor for the PPO Ratio. | **Static** (Per Iteration) |
| **Reference** | $\pi_{ref}(s_t) \rightarrow P(a_t)$ | **The Moral Compass:** The original SFT model. Used for KL Tax. | **Frozen** (Permanent) |
| **Critic** | $V_\phi(s_t) \rightarrow \text{Value}$ | **The Accountant:** Predicts "Expected Future Profit" ($V$). | **Active** (Updates) |

where 
- $P(a_t)$ denotes the probability distribution over actions
    - For generative LLMs: action = token
    

**All four models** 
- start off as copies of the SFT-trained model
    - although the head of the Critic is replaced
- but diverge during PPO
    - different weight update frequencies
    - different Loss functions

**Recall**

The Surrogate Loss for PPO is
$$
J_{\mathrm{PPO}}(\theta) = \mathbb{E}_\tt \left[
\min{} \left(
r_\tt(\theta) \hat{A}_\tt,\;
\mathrm{clip}\left(r_\tt(\theta), 1 - \epsilon, 1 + \epsilon\right) \hat{\advseq}_\tt
\right)
\right]
$$

It will be useful to keep this in my mind as we explore the behavior and purpose of each component;mm

## The Three Policies ($\pi$): Actor, Old, Reference

The **Actor**, **Old Actor**, and **Reference** models all share the same structural signature. 
- and Loss function

When you give them the  state $s_t$ (representing the output sequence of length $t$)
- they output a probability distribution over the vocabulary
- "the next token"

### Actor 

This is the component we are training to solve our task.
- Role: generate the next action/token
- Update frequency: every mini-batch
- "Based on my current training, I want to say *this* token next."
    
The Old Actor and Reference
- serve to *constrain* how far the Actor's policy can deviate
    - Old Actor: from the Actor's policy at the *start of the epoch*
    - Reference: from the original (SFT trained) model

### Old Actor

- Role: serves as an anchor to prevent the updates from deviating too much in this epoch
- Update frequency: every epoch
- "This is what I would have said at the beginning of this training batch."
- Enforced by: the  Clipping term in the PPO Surrogate Loss

$$
\min{} \left(
r_\tt(\theta) \hat{A}_\tt,\;
\mathrm{clip}\left(r_\tt(\theta), 1 - \epsilon, 1 + \epsilon\right) \hat{\advseq}_\tt
\right)
$$
where

$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\rm old}}(\actseq_\tt |\stateseq_\tt)}
$$

where $\pi_{\theta_{\rm old}}$ is the *Old Actor policy*

    
### Reference

- Role: serves as an anchor to prevent updates from deviating too much from the *original* (SFT trained) model
- Update frequency: never

- "This is what a helpful, safe human-trained model would say."
- Enforced by: KL Penalty
    - mitigates "reward hacking"
        - Reward Hacking: Actor might discover that repeating the word "Excellent!" 500 times tricks the Reward Model into giving it a perfect score
            - even though the output is useless to a human.
    - mitigates "Catastrophic forgetting"
        - loosing the capabilities that were present after SFT training

### Summary of the Three Policies

| Persona | Update Frequency | Purpose |
| :--- | :--- | :--- |
| **Actor** | **Every Mini-batch** | Continuous learning. It adjusts its weights slightly after seeing a small slice of data to maximize the Advantage. |
| **Old Actor** | **Every PPO Iteration** | Stability. It is "refreshed" only after a full set of epochs (e.g., after 4 passes through the buffer) to become the new baseline for the next round. |
| **Reference** | **Never** | Consistency. It remains the original SFT model throughout the entire training process to ensure the model doesn't drift into gibberish. |

---

## The Critic ($V$)

The **Critic** is fundamentally different than the three policy models, as seen by its signature
- same input as the three policy models: state $s_t$ (representing output sequence of length $t$)
- **but** output is a **scalar**
    - representing the return $G_t$ from state $s_t$
    
    The **Return** ($G_t$) at time step $t$ is the sum of all future rewards:
$$G_t = r_t + r_{t+1} + r_{t+2} + \dots + r_T$$

    
That is:
- it is the State Value function
- which we learned in the Intro to RL

The Critic ($V_\phi$) attempts to learn a mapping from the current state to this expected return:
$$V_\phi(s_t) \approx \mathbb{E}[G_t \mid s_t]$$

    

It differs from the three Policy models in Loss function as well
- Three policy models Loss
    - log probabilities (Cross Entropy Loss for next token prediction)
- Critic
    - MSE between return predicted by Value function and true Return
    
$$L_\text{critic} = \text{MSE}(V_\phi(s_t), \text{Returns})$$



## Summary: The "Sync" Challenge

Because the **Actor** and **Critic** are the only two personas that are actively learning, they are the only two that can "fall out of sync." 

1. **The Reference** is a fixed point in the past (The SFT baseline).
2. **The Old Actor** is a fixed snapshot of the current iteration.
3. **The Actor and Critic** are in a constant race. As the Actor changes its strategy, the Critic must "sprint" to update its Value Map to match that new strategy.



# The Three Policies In Depth

## Actor and Old Actor

**The Actor vs. Old Actor "Drift"**

Here is the interaction between Actor and Old Actor
1. **Step 0:** At the start of a PPO iteration, **Actor == Old Actor**.
2. **Step 1:** We update the **Actor** on the first mini-batch. Now it is slightly different from the **Old Actor**.
3. **Step 2:** We calculate the "Probability Ratio" ($r_t$) between the **Actor** (new) and the **Old Actor** (frozen snapshot).
4. **Step 3:** If the **Actor** drifts too far from the **Old Actor** (e.g., > 20%), the PPO Clip kicks in and stops the update.
5. **End of Iteration:** Once we've finished all epochs, we finally update the **Old Actor** by copying the current **Actor** weights into it. The cycle repeats.

**Why limiting drift of Actor from Old Actor matters for both the Actor and the Critic**

For efficiency, the training episodes for the Actor
-  are *not generated* real-time
- they are generated at the *beginning of the epoch** by the **Old Actor**
    - these episodes are called *rollouts*
    - stored in an "Experience Buffer"
    - used by the Actor for training
- the Experience Buffer may be re-used for multiple epochs

What this means is that
- the training distribution (Old Actor)
- may become different than the distribution learned by the Actor
    - after inter-epoch gradient updates
    
Temporary violation of the Fundamental Assumption of Machine Learning !

So typing the Actor to the Old Actor makes sense in order to limit the drift.
    

The Critic has the same issue
- its goal is to provide feedback (create value function) to guide the Actor
    - for the current Actor (new distribution)
- but the Actor is one-step ahead of the Critic in gradient updates !

So the Critic's feedback is outdated (out of sync) with the Actor
- tying the Actor to the Old Actor
- **also** limits how far out of sync the Actor and Critic can become



### Why RL is "Special" (and harder than Supervised Learning)

In Supervised Learning
- we don't worry as much about the distribution shifting mid-epoch
- because the labels are grounded in reality 
    - if the model updates in a way that diverges from the label
    - Future losses and gradient updates will pull it back

In RL
- the **Experience Buffer is a reflection of the Actor's past self**. 
- If the Actor changes too much during an epoch:
    - The **Critic's Baseline** becomes obsolete.
    - The **Advantage Signal** becomes invalid
    - Since the Advantage weights the Gradient updates
        - it may damage the weights by acting as "Noise" rather than "Signal"

PPO addresses this drift via the term


$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\rm old}}(\actseq_\tt |\stateseq_\tt)}
$$

in the Surrogate Loss
- where "new" and "old" refer to "Actor" and "Old Actor"



## Reference

### The KL Penalty: Defining the "Moral Compass"**

The KL Penalty is an intermediate negative reward applied at **every single token** during the generation process. 

It measures how much the current **Actor** ($\pi_\theta$) has deviated from the original **Reference** model ($\pi_\text{ref}$).

**The Mathematical Definition**

The penalty at token $t$ is defined as the KL Divergence between the Actor's probability distribution and the Reference model's distribution:

$$r_{kl, t} = -\beta \cdot \text{KL}(\pi_\theta(a_t \mid s_t) \| \pi_\text{ref}(a_t \mid s_t))$$

In simpler terms, for a specific action $a_t$:
$$r_{kl, t} = -\beta \cdot \left( \log \frac{\pi_\theta(a_t \mid s_t)}{\pi_\text{ref}(a_t \mid s_t)} \right)$$

* **$\beta$ (Beta):** The "Stiffness" of the leash. A high $\beta$ keeps the model very close to the original SFT version.
* **The Log Ratio:** If the Actor makes a word 10x more likely than the Reference model did, the penalty grows.

**Purpose of the KL Penalty: The Three Guardrails**

The KL Penalty serves three critical functions that keep RL training from collapsing:

* A. Preventing Reward Hacking

    The Reward Model can sometimes be "tricked" into giving high reward for nonsensical output

    A model that generates nonsense to reap high reward is said to be "Reward Hacking"


* Preserving General Capabilities (Catastrophic Forgetting)**

    An unconstrained Actor might forget all the capabilities acquired from SFT training.
    
    Thus, we anchor the Actor to the Reference model.


* C. Ensuring "Smooth" Training**

    In the Actor-Critic framework, if the Actor changes too fast, the Critic can't keep up (the "Sync" problem). 
    
    The KL Penalty ensures that the Actor only explores the space "near" its known safe starting point, making the learning process stable.

# The Critic in Depth

Reinforcement Learning is designed around the goal of
- *positively reinforcing* "good" actions
- *negatively reinforcing* "bad" actions

This is a challenge when the task has sparse rewards
- single end-of-episode reward

The *Credit Assignment* problem is to
- decompose the single reward
- into rewards per action in the episode
- to facilitate positive/negative reinforcement



The Critic solves the Credit Assignment problem
- by creating a Value function
    - expected return to go for each state
        - it is an expectation because of possibility of stochastic Agent and Environment

The Value for a state
- is an *estimate* of expected return
- acts as a "baseline" for what the anticipated  expected return to go will be

We compare the *actual return* from a state to the baseline
- this is called the **Local Advantage**

The Policy Gradient is *weighted by advantage*
- to selectively strengthen or weaken specific tokens in a sequence

**Key Insight:**
The Critic turns a **Sparse Reward** (one number at the end) into a **Dense Signal** (a number at every token).

The Critic can also be used for tasks with *dense* rewards
- reward at every step

Consider two states in the episode
- receiving the same reward for the chosen action
- but with **different**  state Value ("baseline" for anticipated reward)

Even though the rewards were identical
- one is a positive surprise (relative to baseline)
    - should be positively reinforced
- one is a negative surprise
    - should be negatively reinforced


Subtracting a baseline from the return
- results in an advantage in units "distance from baseline"
- so advantages are naturally zero centered (if baseline is average)
- reduces variance of gradient updates; stabilizes training

# Computing the Advantage : The Journey of the "Surprise" 

## Simplifying assumptions

To isolate how PPO assigns credit to specific steps/tokens
we start with these simplifying assumptions.

These assumptions allow us to 
- see how a single final score is "backpropagated" to specific intermediate actions 
- without the noise of time-decay or mid-process penalties.

**Assumptions**

* Sparse rewards

    Single end-of-episode reward with magnitude $1$


* No Discounting

    $\gamma = 1$


* Trace Decay 

   $\lambda = 1$
   
   We define $\lambda$ as the parameter that determines 
   - how much we "trust" the full trajectory. 
   
   At $\lambda=1$, we trust the **Total Episode Reward** completely to define our Advantage.


* We omit the KL Penalty

    This is implemented as an intermediate reward (every step) so we omit it to preserve the assumption of sparse rewards.

## Local Surprise (TD Error)

The "Local Surprise" $\delta_t$ is mathematically defined as 
$$\delta_t = V(s_{t+1}) - V(s_t)$$

That is, the local surprise is
- how much the action taken at $s_t$
- changed (for the better or worse) the return

Interpretation of the surprise:

1.  **If $\delta_t > 0$:** The Actor did something **better** than the Critic expected. The Actor is rewarded.
2.  **If $\delta_t < 0$:** The Actor did something **worse** than the Critic expected. The Actor is discouraged.
3.  **If $\delta_t = 0$:** The Actor did exactly what was expected. No "Surprise" means no "Advantage," so no change to the weights.

We can relate the Local Surprise to the TD Error used in Value-based models.

Recall, we update our estimate for $V(s_t)$ from episode $k$ to episode $k+1$

$$
\begin{array} \\
V_{k+1}(s_t) & = & r_t + \gamma V_k(s_{t+1}) & \text{Immediate reward } r_t \text{ plus discounted return of next state} \\
V_{k+1}(s_t) - V_{k}(s_t) & = & r_t + \gamma V_k(s_{t+1}) - V_{k}(s_t) & \text{algebra} \\
& = & V_k(s_{t+1}) - V_{k}(s_t)  & \text{simplifying assumptions } r_t = 0, \gamma = 1 \\
& = & \delta_t & \text{TD error} \\
\end{array}
$$

Under our simplifying assumptions
- Sparse rewards: $r_t = 0$ for $t \lt T$
- No discounting: $\gamma = 1$

the Local Surprise is identical to the TD Error used in Value-based models.


The Local Surprise has two components:

* **The Reality:** The immediate reward ($r_t$) plus the Critic's estimate of the *next* state ($V(s_{t+1})$).
* **The Expectation:** The Critic's estimate of the *current* state ($V_k(s_t)$).


> **Translation:** "The value I see now ($r_t + V_{next}$) minus the value I predicted a moment ago ($V_{current}$)."

## Advantage $\hat{A}_t$ = Total surprise

Once the episode ends and we see the real reward ($R_\text{final}$), 
we calculate the Total Surprise. 

Mathematically, 
- the Total Surprise (Advantage)
- is exactly the sum of every Local Surprise from that point forward:

$$\hat{A}_t = \delta_t + \delta_{t+1} + \dots + \delta_T$$


 This is the "Reality Check." 
 - It’s the difference between what we *thought* would happen at token $t$ 
 - and what actually happened at the finish line.
 
Under our implifying assumptions:
$$\hat{A}_t = R_\text{final} - V(s_t)$$


# Using the Advantage: The Gradient Update

In the Actor's loss function, 
- we use the **Total Surprise** ($\hat{A}_t$) 

as the multiplier for the gradient:

$$\nabla_\theta L \approx \hat{A}_t \nabla_\theta \log \pi_\theta(a_t | s_t)$$

* **If $\hat{A}_t$ is positive:** We "push" the Actor to make that choice more likely.
* **If $\hat{A}_t$ is negative:** We "pull" the Actor away from that choice.

---

## Numerical Example: "The Brilliant Start, The Tragic End"

Imagine a model writing a 3-token code snippet.
* **Start:** Critic expects a mediocre result: $V(s_1) = 0.5$.
* **Step 1:** Actor writes a brilliant function header. Critic is impressed: $V(s_2) = 0.9$.
* **Step 2:** Actor writes a standard line. Critic stays steady: $V(s_3) = 0.9$.
* **Step 3:** Actor makes a typo that breaks the code. Reward Model gives: $R_{final} = 0.2$.

**Calculating Local Surprises ($\delta$)**

1.  **$\delta_1$ (The Genius):** $V(s_2) - V(s_1) = 0.9 - 0.5 = \mathbf{+0.4}$
2.  **$\delta_2$ (The Neutral):** $V(s_3) - V(s_2) = 0.9 - 0.9 = \mathbf{0.0}$
3.  **$\delta_3$ (The Blunder):** $R_{final} - V(s_3) = 0.2 - 0.9 = \mathbf{-0.7}$

Calculating Advantage ($\hat{A}_t$) for the Update:
 
* **$\hat{A}_1$ (For token 1):** $\delta_1 + \delta_2 + \delta_3 = 0.4 + 0.0 + (-0.7) = \mathbf{-0.3}$
* **$\hat{A}_3$ (For token 3):** $\delta_3 = \mathbf{-0.7}$

**The Conclusion:** 

Even though the whole episode was a "failure" ($0.2 < 0.5$), 
- the first token only gets a small penalty ($-0.3$) because the Critic "defended" it with a positive local surprise. 
- The third token takes the massive hit ($-0.7$).

# Comparison to GRPO


In GRPO, for each prompt, we form a group of size $G$ completions.

Here is the GRPO advantage for the $i$-th completion in the group

$$A_i = \frac{R_i - \text{mean}(R_\text{group})}{\text{std}(R_\text{group})}$$

Every token $a_{i,t}$ within that completion $i$ will use this same $A_i$ value during the policy gradient update.

Contrast this to PPO Advantage
- different advantage per token

The GRPO Philosophy: "A Rising Tide Lifts All Tokens"

In GRPO, if an Actor produces a completion that earns a high reward ($R=1.0$) compared to the group average ($\mu=0.5$), 
- the math tells the model:
> "Everything you said in this specific completion was good. Strengthen the weights for **every single token** in this sequence."

The drawback of all tokens sharing equal credit
- a "bad" token can hitch a ride on a "good" sentence. 

* *Example:* If the model writes a brilliant essay but includes one typo, 
- GRPO will still "reward" the typo because the overall essay score was high.

## The Strength of GRPO

By removing the Critic, you save **50% of your VRAM**. 

This allows you to:
1. Increase the **Group Size** (e.g., compare 64 versions of the same prompt).
2. Use that extra memory to train a much **Larger Actor**.

## The Strength of PPO

Even in the extreme case where
- there is a single end-of-episode reward
    - no intermediate rewards
    
the PPO Advantage is able to
- assign credit (decompose the Episode Reward)
- to each step/token

rather than equally distributing it to the steps
- as is done in GRPO

**This is surgical credit assignment.**

**Surgical vs. Holistic Credit Assignmen**

| Feature | PPO (Surgical) | GRPO (Holistic) |
| :--- | :--- | :--- |
| **The Baseline** | **The Critic's Prediction:** A unique value $V$ for every single token. | **The Group Mean:** A single average score $\mu$ for the entire group of 16-64 completions. |
| **The Advantage** | **Token-Level:** Calculated via the sum of TD Errors ($A_t$). Each word gets its own unique "Surprise" score. | **Sentence-Level:** Calculated as the standardized score ($A_i = \frac{R_i - \mu}{\sigma}$). Every token in a "winning" sentence gets the same reward. |
| **Computation** | **Heavy:** Requires a separate Critic model (often as large as the Actor) running alongside it. | **Light:** No Critic model. You just run multiple Actor completions in parallel. |

##  Summary; PPO vs GRPO Comparison Table

| Feature | PPO (Proximal Policy Optimization) | GRPO (Group Relative Policy Optimization) |
| :--- | :--- | :--- |
| **Advantage Source** | **Learned:** The Critic's $V(s)$ prediction. | **Calculated:** Group Mean/Std Dev of rewards. |
| **Model Count** | 4 (Actor, Critic, Ref, Reward). | **2-3** (Actor, Ref, Reward). |
| **Precision** | **Surgical:** Can assign credit to a specific word. | **Coarse:** Primarily judges the whole sequence. |
| **VRAM Usage** | **Very High:** Value Head + Critic gradients. | **Low:** Reverts to standard SFT structure. |
| **Stability** | High (if Critic is well-trained). | High (uses the group as a natural baseline). |

# Practical PPO: removing the simplifying assumptions 

To smooth the presentation of concepts for Advantage, we made several simplifying assumptions.


To transition to "Production-Grade" PPO, we re-introduce the following:

## Re-introducing the Discount Factor ($\gamma < 1$)

We multiply future surprises by $\gamma \approx 0.99$.

* **Effect:** The Advantage at token $t$ cares *more* about the surprises that happen immediately after it and *less* about what happens 500 tokens later.

## Re-introducing the KL Penalty (Intermediate Rewards)

The KL Penalty is 
- an intermediate negative reward 
- applied at **every single token** during the generation process. 

It measures 
- how much the current **Actor** ($\pi_\theta$)
- has deviated from the original **Reference** model ($\pi_\text{ref}$).

Re-introducing the Penalty as an intermediate reward also allows us to accommodate non-penalty
intermediate rewards.

The penalty $r_{kl, t}$ at token $t$ is defined as the KL Divergence between 
- the Actor's probability distribution 
- and the Reference model's distribution:

$$r_{kl, t} = -\beta \cdot \text{KL}(\pi_\theta(a_t \mid s_t) \| \pi_\text{ref}(a_t \mid s_t))$$





In a world with KL Penalties, the "Surprise" at every token is now a balance of **Behavior** and **Value**:

$$\delta_t = \underbrace{(r_{kl, t} + \gamma V(s_{t+1}))}_{\text{The New Reality}} - \underbrace{V(s_t)}_{\text{The Prediction}}$$

Every token the Actor produces now "costs" a little bit of KL Tax. 

To have a positive Advantage
- the Actor must produce a token 
- that is so good
- it justifies the cost of deviating from the Reference model.

We add a small negative reward $r_{kl}$ (KL Penalty) at **every** token 
- to prevent the model from becoming "too weird."

$$\delta_t = (r_{kl} + \gamma V(s_{t+1})) - V(s_t)$$


### Updated Numerical Example (with -0.1 KL Penalty per step):

The Local Surprise must now account for the "reward in hand":
$$\delta_t = r_t + V(s_{t+1}) - V(s_t)$$

* **$V(s_1)=0.5, V(s_2)=0.9, V(s_3)=0.9$**
* **$r_1=-0.1, r_2=-0.1, r_3=0.1$ (Final 0.2 minus 0.1 KL)**

1. **$\delta_1 = -0.1 + 0.9 - 0.5 = +0.3$** (Brilliance, slightly taxed)
2. **$\delta_2 = -0.1 + 0.9 - 0.9 = -0.1$** (Standard move, costs more than it earns)
3. **$\delta_3 = 0.1 - 0.9 = -0.8$** (The blunder, even more painful)

**The Identity Holds:**
The Total Advantage $\hat{A}_1$ is still the sum of future surprises:
$$\hat{A}_1 = 0.3 + (-0.1) + (-0.8) = -0.6$$
This is exactly the same as: **(Actual Total Rewards) - (Initial Expectation)**
$$( (-0.1) + (-0.1) + 0.1 ) - 0.5 = -0.1 - 0.5 = -0.6$$


## Re-introducing Trace Decay: Generalized Advantage Estimation (GAE - $\lambda < 1$)

Under our simplifying assumptions
- Total Surprise was a simple sum of present and future local surprises


$$\hat{A}_t = \delta_t + \delta_{t+1} + \dots + \delta_T$$

But we may have less confidence in the Critic's estimate ("Critic Noise") of local surprises farther away in the future than those closer.

We apply a "Trace decay" factor $\lambda$ to reflect this uncertainty.
- mitigates compounding of errors in distant estimates


$$\hat{A}_t = \delta_t + (\gamma\lambda)\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} + \dots$$



This is called the *Generalized Advantage Estimate (GAE)**.

**Aside**

We also reintroduced the "time decay" factor ($\gamma$) favoring near term rewards over distant ones.


# Comparison of Credit Assignment Methods

In our Unified Policy Gradient formulation, it was the Advantage calculation that mainly
distinguished between RL methods.

We now compare some of these Advantage calculations.


Note that the Local Surprise, with Time Discounting applied, is written as

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

## REINFORCE ($\lambda = 1, \text{No Critic}$)

* **Formula:** $\hat{A}_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$
    - this is the formulation for dense rewards
    - replace the sum with the episode reward $R(\tau)$ for the case of sparse, end-of-episode reward 
* **Intuition:** "The end justifies the means." 
* **Pros:** Guaranteed to find the truth eventually.


The issue with REINFORCE is that future rewards are *random variables*
- depend on future actions of the Agent and responses of the Environment

So the Advantage is a random variable
- Unbiased because it is the actual truth of what happened in the episode
- High Variance because it looks at rewards of each step until the end of the episode
    - so it is the sum of random variables, each with a variance
    - thus variance increases with length of episode

High variance Advantage (which weights Gradient updates) leads to less-stable training.

##  1-Step Actor-Critic ($\lambda = 0$)

We did not actually study this method.
- it is a one-step version of what was used in PPO under simplifying assumptions
- equivalent to GAE with $\lambda = 0$ (to kill future terms)

It uses Local Surprise rather than Total Surprise

* **Formula:** $\hat{A}_t = \delta_t$
* **Intuition:** "I only care about what happens in the next second."
* **Pros:** Highly stable; creates a very smooth gradient.


Since this method looks only a single step into the future, it is
- High bias: information is completely local; not dependent on what actually will happen in subsequent steps
- Low variance: depends on only a single random variable

The low variance Advantage leads to stable gradient updates during training.

## GAE ($0 < \lambda < 1$)

GAE improves upon the 1-Step Actor-Critic by using *Total* surprise rather than *Local* Surprise.
- **Verifies** the current surprise by looking at future outcomes.

It further improves upon it by adding the Trace Decay ($\lambda$) to account for uncertainty.
- $\lambda$ typically; 0.95

* **Formula:** $\hat{A}_t = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}$
* **Intuition:** "I trust the near-future, but I'm skeptical of the distant-future."

GAE gets closer to low bias, low variance
- Lower bias than 1-Step Actor-Critic: because it looks at all future steps
    - but $\lambda \lt 1$ introduces some bias
- Lower variance than REINFORCE: although it looks at all future steps, the future terms are weighted by $\lambda \lt 1$

**GAE was an important, fundamental improvement in RL**

- by avoiding difficult trade-off decisions between bias and variance
    - it simplified training
    - less hyper-parameter search and tuning
    - training became an "engineering" process rather than an "experimental" challenge
- stabilized training
- facilitated long horizon tasks (long sequences)
    - since variance compounding was controlled

**Reference**

[High-Dimensional Continuous Control Using Generalized Advantage Estimation](https://arxiv.org/abs/1506.02438)


# Pseudo-code for PPO

To understand how PPO actually trains, we must look at the two distinct "gears" of the algorithm: 
- the **Outer Loop** (Experience)
- the **Inner Loop** (Optimization)


    # --- INITIALIZATION ---
    # Actor (π_θ), Critic (V_φ), Reference (π_ref), Reward (RM)

    # --- THE OUTER LOOP (The Experience Phase) ---
    # Goal: The model "acts" in the world and gathers feedback.
    for iteration in range(max_iterations):

        # 1. ROLLOUT: Actor generates a batch of completions.
        # We save the 'logprobs_old' and 'values_old' from this specific version.
        sequences, logprobs_old, values_old = actor.generate(prompts)

        # 2. EVALUATION: The external and internal judges speak.
        rewards_terminal = reward_model.score(sequences)
        kl_penalties = compute_kl(sequences, actor, ref_model) # Intermediate rewards

        # 3. SURGICAL PREP: Calculate the Advantage (Total Surprise)
        # This uses the telescoping sum of local surprises (δ)
        # δ_t = (r_kl + V_next) - V_now
        advantages = compute_gae(kl_penalties, rewards_terminal, values_old)
        returns = advantages + values_old # The 'Target' for the Critic

        # --- THE INNER LOOP (The Optimization Phase) ---
        # Goal: Use the surgical feedback to refine the brain.
        for epoch in range(ppo_epochs):
            for batch in shuffle(sequences, advantages, returns, logprobs_old):

                # A. ACTOR UPDATE: "The Performance Review"
                # Compare current probs to logprobs_old.
                # If Advantage is positive, push probabilities up (with PPO Clipping).
                policy_loss = actor.compute_loss(batch, advantages, logprobs_old)
                actor.step(policy_loss)

                # B. CRITIC UPDATE: "The Accounting Audit"
                # The Critic tries to minimize the error between its guess (V)
                # and the actual outcome (returns).
                value_loss = critic.compute_mse(batch.values, batch.returns)
                critic.step(value_loss)

        # After the Inner Loop, the Actor is slightly better.
        # It becomes the 'Old' model for the next Outer Loop iteration.

**Code highlights**

| Phase | Purpose | Concept Applied |
|:--- |:--- |:--- |
| **Outer Loop** | Data Collection | **Reference Model** calculates KL penalties to keep the model human-like. |
| **Advantage Calculation** | Credit Assignment | **Telescoping Sum** turns final rewards and KL taxes into a "Surgical" signal for every token. |
| **Inner Loop** | Weight Update | **Old Model** ensures the Actor doesn't change too much in a single step (PPO Clip). |
| **Critic Update** | Learning the Map | The **Critic** learns to be a better "Accountant" by comparing its guesses to the actual results. |

## Why is this so expensive?

By looking at this loop, you can see the **VRAM footprint**:

1. **Actor:** Being updated (Gradient storage required).
2. **Critic:** Being updated (Gradient storage required).
3. **Reference Model:** Constant forward passes for KL.
4. **Reward Model:** Constant forward passes for scores.

# HuggingFace TRL

Here is  how **TRL** handles the architectural heavy lifting for the Critic model.



## The "Heavy Lifting": How TRL Builds the Critic

In practice
- you don't need to manually perform surgery on your model’s neural network layers
    - e.g., replace the head of the SFT-trained model to become the Critic
    
Libraries like HuggingFace **TRL** (Transformer Reinforcement Learning)
- automate the process 
- of turning a standard **SFT (Supervised Fine-Tuning)** model into an **Actor-Critic** pair.

### The `AutoModelForCausalLMWithValueHead` Wrapper
The primary tool TRL uses is a specialized wrapper class. Instead of loading a standard language model, you load the model using this class:

```python
from trl import AutoModelForCausalLMWithValueHead

# This loads the SFT model and automatically attaches a "Value Head"
model = AutoModelForCausalLMWithValueHead.from_pretrained("your-sft-model-path")
```

**What the Library Does Automatically:**
* **The Backbone:** It loads the full pre-trained transformer (the "Body").
* **The Actor Head:** It keeps the existing Language Modeling head (the "Voice") that predicts the next token.
* **The Critic Head (The Surgery):** It identifies the final hidden layer of the transformer and attaches a **new, randomly initialized linear layer**.
* **The Output:** This new layer is a "Scalar Head"—it takes the high-dimensional vector from the transformer and reduces it to a single number: the **Value $V(s)$**.

###  Shared vs. Separate Backbones
One of the most important "heavy lifting" decisions TRL makes is **Weight Sharing**.

* **Shared "Trunk":** By default, the Actor and the Critic share the same transformer body (e.g., the 7B parameters). They only diverge at the very end with their specific "heads."
* **VRAM Efficiency:** This saves massive amounts of memory. You don't have to load two entire LLMs; you only load one LLM with two small "caps" on top.
* **The "Tug-of-War":** Because they share a body, the Actor wants to update the weights to be a better writer, while the Critic wants to update them to be a better judge. They are essentially "tugging" on the same set of weights.

###  The "Cold Start" of the Accountant
Even though TRL handles the architecture, it cannot give the Critic "knowledge" instantly.

* **The Actor:** Starts **"Smart."** It already knows how to speak because it was trained during the SFT phase.
* **The Critic:** Starts **"Blind."** Because the Value Head was randomly initialized, its first few guesses for $V(s)$ will be total nonsense.

**The Result:** During the first few batches of PPO, the `PPOTrainer` focuses heavily on the **Value Loss**. The Critic has to "sprint" to catch up to the Actor's current performance level so it can provide a stable baseline for the Advantage calculation.

###  Comparison: PPO vs. GRPO Architecture
This architectural overhead (and the VRAM cost of the Reference Model) is exactly what **GRPO** (Group Relative Policy Optimization) seeks to eliminate.

| Feature | PPO (via `trl`) | GRPO |
| :--- | :--- | :--- |
| **Model Type** | `AutoModelForCausalLMWithValueHead` | Standard `AutoModelForCausalLM` |
| **Extra Heads** | Yes (Scalar Value Head) | **None** |
| **Backbone Tug-of-War** | Actor vs. Critic | **None** |
| **Accountant Required?** | Yes (The Critic) | No (Group Statistics) |

In [2]:
print("Done")

Done
